## Setup

In [1]:
from agent_base.core.messages import *
from agent_base.providers.anthropic import (
    AnthropicLLMConfig,
    AnthropicMessageFormatter,
    AnthropicProvider,
)
import anthropic
from dotenv import load_dotenv
from pprint import pprint

load_dotenv("../../../.env")

formatter = AnthropicMessageFormatter()
client = anthropic.AsyncAnthropic()
anthropic_provider = AnthropicProvider(
    client=client,
    formatter=formatter
)
# generate() / generate_stream() return a ProviderTurn (providers.md §2.1):
# the canonical assistant Message rides on `.message`; `was_cancelled`,
# `partial_error` and provider-private `stream_bookkeeping` ride alongside.

## Provider Round Trip Tests

Canonical Message -> format -> anthropic client -> parse -> `ProviderTurn` (`.message` is the canonical assistant `Message`)

### Basic - No Tools

In [2]:
canonical_message = Message(
    role=Role.USER,
    content=[
        TextContent(
            text="Hello, world!"
        )
    ]
)
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[canonical_message],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
    )
)
pprint(res.message.to_dict())

{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EuwCCm0IDhgCKkBJNtgwmx8HMnkwuRxPuJNWVR3fxqhTBi1dS4oaGR8uPzgDBp+bBmvIKYerR9TcDfBkIoy8kG8QcPkfSuoYBoWXMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgzkJFAgCXdXk/Ugb8MaDL0LRxZSNTcNz3jwhSIwtn7gumGy+/U4TnT0jnftADFXOCbjVQQVoR18IWf6gAh7MrtCrKkGWRM448PLDkXcKqwBsLxPXPx9fo1lA96yLvzBHfFagHYz/SqOx+WHFzB1vcVRvfpcTwSo2ySQrFBL9QzgY0Qha82dq76gucSdYlEBgwQg2zwRiyPMwZLyNIVv1hmbz5fc9NJ2SSJ+uJKH1ZAUKXQmMEseaByhIGt9/09ABv0AehpI5brmv5pu6PAhoc7NIRsjyAq6QmDW6uTw5Y8GZdbPJQzlZRY66SqM7dleUQ/xE51PP/4ROMF0oxgB',
              'thinking': 'The user has greeted me with a simple "Hello, '
                          'world!" - a classic programming phrase. This is a '
                          'friendly greeting and I should respond warmly and '
                          'helpfully.'},
             {'content_block_type': 'text',
              'kwargs': {},
              'text': "Hello! 👋 It'

### Server Tools, Betas, Context Management

In [3]:
res = await anthropic_provider.generate(
    system_prompt="Be brief.",
    messages=[Message.user("What is Anthropic's latest model?")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[
            {"type": "web_search_20250305", "name": "web_search", "max_uses": 3},
        ],
        beta_headers=["files-api-2025-04-14"],
    )
)
pprint(res.message.to_dict())

{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EvUDCm0IDhgCKkASvoheBxB+Vq+vue9yUgT/UoqhPc0y4zqt16vswYfKoACNAMEhshc1RIATxHpZpwMVFs35PowjXseY/LXMVmEDMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgxQPs3e99DNallmptYaDEVzoA5CuWn5qKdFpiIwFD/YT9ThyszbQTrvLtDZFECyMPqC12hTrxVHFI36hphbv8FaaI8rBgm/cwvetPDpKrUCy7v/qSgtN71w1JHVxdgQVFUaOSnsNQ46BZiMp/Vov5VpouEhHAY7DFnbsATFIAaIKADtcYzkR/WFbz5YR1fgQGEgYZO0hxDGiKqrYoW7SXkc7QeMwtPf6FoTPb1rn7KYiQzQ1caoSTnb8CCUpdjRlPs0y1vhM25aFXOFIb+D7OwAP6Xo2wXDo8WfJr/GQgW2ddAZqVDTrM2FKGzY22vkiV4Gdqy7kPc7dqMjQhDHiJp1Qeo41UhZ9xIZHJkC6QBF/3uWecAx6Ob+9QOLtJQLZjo4Ua0ycz5Ew2e7uXxzXUsx972yG4T+ZgqzYva6IXIUu2D4BW/4ncT8cxQh+kpk/22BQJ4WnlXpVE/q0EePqcOBGdtrXfPgBPjrfwWOERQK+QRH2EH0ASY4FNHwNtQDkiazjbddGAE=',
              'thinking': "The user is asking about Anthropic's latest model. "
                          "My knowledge cutoff is from April 2024, and today's "
                          'date acco

### Client Tools

In [4]:
from agent_base.tools.tool_types import ToolSchema
from agent_base.core.types import ToolResultContent, ContentBlockType

schema = ToolSchema(
    name="get_weather",
    description="Get current weather for a city.",
    input_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
)

msg = Message.user("What's the weather in Paris?")
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[msg],
    tool_schemas=[schema],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.message.to_dict())

tool_call = next(b for b in res.message.content if b.content_block_type == ContentBlockType.TOOL_USE)
tool_result_msg = Message(role=Role.USER, content=[
    ToolResultContent(tool_name=tool_call.tool_name, tool_id=tool_call.tool_id, tool_result="Sunny, 22°C")
])
res2 = await anthropic_provider.generate(
    system_prompt=None,
    messages=[msg, res.message, tool_result_msg],
    tool_schemas=[schema],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res2.message.to_dict())

{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EooDCm0IDhgCKkDdSKuuPqsUc/0Q0t5HH1ODJ/P+hEau1kSjfV99iDzVsCMI7dIH+y/YLFiNYqho30k1dF6ZwqplKv03gRNWmoZQMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgyFQXmltFgxZVwvrAsaDKz36J7Z3z9gF1BC9iIwLzMB6BMG3R+6L+4coTFkwIdiC5Ofmb3KFNm1+ho5ym8uF3JluIWRAv46CFF4+R9iKsoBkH/UZgz9FtsT1sKMLWf+8sq8eadzJGsJdHQ0HTGyFoEVNApC4hqKamzeYa/3j0Sj/NtH80a19vkmSSRfIDxC1XP8VyzH1TDdlNLVLhuGuLUUTnV8P8fnKxXAxZ8hGQ18I1C5BUkvJg8/NuDgMY//5jXnSKNXS3k7r8CkTXMv3U+bUCyJNNns2sy6pJFVnSOc+EdOeZPNsbPjKhHusmGXcpGOcG7Mf5bkPN6OpXYut970TMIcZZhf0BBGOWVzfyrLjBtFtjuYwWsichgB',
              'thinking': 'The user is asking for the weather in Paris. I have '
                          'a function called `get_weather` that takes a city '
                          'name as a parameter. I should call this function '
                          'with "Paris" as the city parameter.'},
             {'content_block_type': '

### Server Side File Generation

In [5]:
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message.user("Write and run a Python script that prints 'hello world'")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        beta_headers=["code-execution-2025-08-25", "files-api-2025-04-14"],
    )
)
pprint(res.message.to_dict())

{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EsYDCm0IDhgCKkCnso1K8B+hIHCN+sDwwe6gZ3mOP/SYpg4Sa+q5lVJHFSibBBrmT3XZrlQYTeCmEHSxieFSCSM9LPROVK6LeYyCMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgwmMtjS2VgQY451waYaDDnlCuNZcgAhxf1g5SIw5oJo4YJqAarYoh+i5aG9Y3ttNepxfu/BzxCuNNy2HWMJO8FZON4Vl0Gs4xLzfh4CKoYCtLdZLPF87CIuEKEhhLzzo4EJkxnaXwsBGmJ8roZduS0Ie4MW3ue7XJLM4BpUTxTN0XmEFuPbjY6VJl8K1QAQu37Fv+ARpPJmFvXRDb9AIYCZgReLb7RL9/MZljDaRvzxCHn3RANGCz7pS+Fynjhgv7OX3nq8juWWOw5EIzv/dStRfnYClIDWzya2pqJ+SdkuM9+Cu6JR7HggyCKT9ive3lUCXOsltxOLjl3t0FWsby19PKkn0x4RfgiR9AzHHeYE0LSKT7rGSNbKSOLpknhKatZlEbnO59u1kUUWuO4jJ6jBOTh5BY7hmNcmASXT2+zRIM/zahzpDYXqRtEGc+IQfquqn3qZQBgB',
              'thinking': 'The user wants me to write and run a Python script '
                          "that prints 'hello world'. I should:\n"
                          '1. Create a Python script using the '
                          'text_editor_code_execu

### File Read Tools

In [6]:
from agent_base.core.types import DocumentContent

# For text/plain documents the canonical layer passes `data` through verbatim
# as the document text (formatters: base64/text + text/plain -> {type: "text"}).
doc_text = "The speed of light is approximately 299,792,458 meters per second."

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type="text/plain", source_type="text", data=doc_text),
        TextContent(text="What speed is mentioned in this document?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.message.to_dict())


{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EuADCm0IDhgCKkAAv1gbZgvOCDPdlQI06ytaOar/Nah2dCBgJ5TnV2nsCowT1M48V/exHVAekpXDj4zk1rrXJ4cW+bzdsZxWQkUBMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgwg7FAZ8X3+7De+gSoaDGLK6a2rWLqFAOAXjyIwXA73yRs7nbfO1PvXCc6Z32pqnp3l1tvBN1jxGw9dh/v7/iOm9C97OxTs1W9lCcpZKqACXMZOoi8R8ztEwLNhR6sGEovW9r9ReIEPTGaQ7S3kyPHtqRteaPrUTPuJsWiW5CvozqXD64x0oqhLcr2vUDszdu4iZixsG+U0IOnRe7LUUzHHHmWDPkoqLoTz5JTcpWQUmzLu6WcrIKVY1m3Tlbht2g5nM07rH4skpJ0Uva7MBPCTyw0ehHVaAleX0yd86pVWf0awJt+wJ+UgfF49a0yjeGG03QlAm+FcZco2Xh+dxv+mfM/t20lFojTNh6PV23WPdkFqAXJl3QxJilmSyTKqLoTuQ8Z6zeoO6+QRzSpJcZ1n16iDntoqxGROqjRF2BCqcfGodEYvPdT7s+mvn+CM65DG8xTRwiU11SRY9pyYQnKUUD1Vy/b7/oS7lPgkKNKOGAE=',
              'thinking': 'The user is asking what speed is mentioned in the '
                          'document. Looking at the document content, it '
                          'states "The speed of light is approximately 

### Container Uploads

In [7]:
from agent_base.core.types import AttachmentContent

file_content = b"print('Hello from uploaded file!')"
file_response = await client.beta.files.upload(file=("script.py", file_content, "text/x-python"))

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        AttachmentContent(
            filename="script.py", media_type="text/x-python",
            source_type="file_id", data=file_response.id,
        ),
        TextContent(text="Read the uploaded file and describe what it does."),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        # container_upload blocks are only valid when a code execution tool
        # is enabled on the request -- the upload lands in that container.
        server_tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        beta_headers=["code-execution-2025-08-25", "files-api-2025-04-14"],
    )
)
pprint(res.message.to_dict())

{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EuoCCm0IDhgCKkAe0V2ANdMrxO7nQizxlB6TeOoiFhQgsm7i0TfhBEnFGr8dLHBqMEvByelwoayiFcZtNdWG766clepleOFzGc26MhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgybkcYU/DvVMHea2kAaDBg5vLGx7nABhqFBryIwBPV5X8zurW3ae9MIOjmSPrxVAri7w6eOioGIpQyKBxLZOUXfDkBUNeciQm1leMHeKqoBSU+SugtvP1I7S3Wys0t/zx0ygCKLZxEJbf7O9MyKN7AC9Eg4Z8569eymuFbBmBH43n08B3ij+kcBSDhj/FJ2s9GTo5Y4TTJX99627Oc52t80VjVjvRBR3LLS1y9uVzMoI10QPr/y0Y4x2y1xRo7rmv9BFVoVBcTySUwS/16aFgH4Ffy4mB5Pp0I+NKTQQEZi7mY9go4oKO8DaXfMu5X2+ptKp9wqhHste4UYAQ==',
              'thinking': 'The user has uploaded a file called "script.py" and '
                          "wants me to read it and describe what it does. I'll "
                          'use the text_editor tool to view the file content.'},
             {'content_block_type': 'server_tool_use',
              'kwargs': {},
              'tool_id': 'srvtoolu_01C9yLuJvQHrADkEgeSD

### Web Search with citations

In [8]:
res = await anthropic_provider.generate(
    system_prompt="Be brief. Cite your sources.",
    messages=[Message.user("Search the web for the current population of Tokyo.")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )
)
pprint(res.message.to_dict())

# Citations ride on the canonical TextContent: raw wire dicts in
# kwargs["citations"], typed (serialized) objects in kwargs["canonical_citations"].
for block in res.message.content:
    if block.kwargs.get("canonical_citations"):
        pprint({"citations": block.kwargs["canonical_citations"]})


{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EokECm0IDhgCKkAXdNPCvob1j8HZJ70FSjRa+Z6N+j/UA5jMCYJfvQrusV6a5COzR7cPsigYKK1jrl6Js/nqMbGUOxJhtCSOpyt1MhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgzGzK8JI07AkQ14j8kaDCoIFskMY2JFP/AaziIw0SVWWalnbo67RvHd93RokonBdYq7dWXL0VjIT4/OtPz0gOJ6a7LlQO6ruMkGRlbuKskCbco3hRmeuCe2GzBlWDkQmHJAvdMfJ7Zi8+RHskkp+H8/0efl0BErSEi40vKQ1yMeDJGnZV/hoSt/mE4eRlRf6s1f6I6iZ8E7/KpPo/zkzI7L2OtYD9bEAspN7uSPsGohshcTwPAEcK6zBZlmtwQtCLS3GVtLOhiw4qMCyboESpEvq4oWL0ABAIiP7a+Hnf1wij818zR/+QwWtHlC9Z2r1aZNlyfHjaYhi8Dt9mibYcjBEyMtXdwee4KZsmbRcHmkyz5PjKNbnWzDXCk1RAptWF2BPvQ7r0ovKyLh8rTnOrRwvjprYBu8sNN5k5Mqt/F0I1GV0rlh7UMoeQ7S8nPEW+mP5UseVldKHfGYi3t5GzYiFWkJmrTQOdBNLwhUlA3prKv0XIjnoSL3HCESMz9jtZAJECnL7ZrnPy1/jVg1MJljSuQoUMqa6J8YAQ==',
              'thinking': 'The user is asking me to search the web for the '
                          'current population of Tokyo. This is a factual '
                

### Document citations

In [9]:
doc_text = "Albert Einstein was born on March 14, 1879 in Ulm, Germany. He developed the theory of relativity."

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(
            media_type="text/plain", source_type="text", data=doc_text,
            # Opt the document into citations (wire: citations={"enabled": true}).
            kwargs={"title": "Einstein bio", "citations_config": {"enabled": True}},
        ),
        TextContent(text="Where was Einstein born?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.message.to_dict())

# Citations ride on the canonical TextContent: raw wire dicts in
# kwargs["citations"], typed (serialized) objects in kwargs["canonical_citations"].
for block in res.message.content:
    if block.kwargs.get("canonical_citations"):
        pprint({"citations": block.kwargs["canonical_citations"]})


{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'Eu8DCm0IDhgCKkDqPdvV3hW9uS4BxCsiuDFDx2zB4Yu82y5USZ2eAi+v1HFRNFmwOmhceqlv0wjikG7f0EXNDroFXZse7RWxZm+GMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgy0O6rLdJAp5W11mVgaDKhyhdSpzquRviAs0yIwKnYIbLEIsE665lMrohJVLBcQEKbWzOdX2UhO+uobFWPKWnaXUMZKraeNv6jaKCAcKq8CppXglEmns4AbdbHFANTVfztOntWeob1BujzOUU/Bdb6M5BPYccAVppcTz8RIrDr0VrK8E0q9hUS+sMP5c2B/CK6NNTB4Dzp1vYaEDyEbPE1+dYKknNQ2DJNf+BfMM71gZ/ohY2uMfU8mEXzVo6AkUspImqRDQAb5H63r+EHY7QGQzUxlZCYMrSdXSFlbM38OgnsASqa7cu3bcNl8G+NKvfryeq0Y+HFfA8io8uF2LJiqvjrISMua0U2uM042GOxDWCgDOUfrhHir8oCynxpJ+QrU26KD7okYB3ioyDHWj0zvLu1MuoiiEMs37IUdkd4zG10l2OrIWzjbBcNmwAFhm7NFLQPnhizHyJ5NFHAPfvuhXjq0i2os+ASKDCrsKjBAyx/JpaOnFScd8VnKAWwCGAE=',
              'thinking': 'The user is asking where Einstein was born. Let me '
                          'look at the document provided.\n'
                          '\n'
                          'From t

### Content Block Citations

In [10]:
doc_a = "Section A: Python is a programming language created by Guido van Rossum."
doc_b = "Section B: JavaScript was created by Brendan Eich and runs in browsers."

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type="text/plain", source_type="text", data=doc_a,
                        kwargs={"title": "Doc A", "citations_config": {"enabled": True}}),
        DocumentContent(media_type="text/plain", source_type="text", data=doc_b,
                        kwargs={"title": "Doc B", "citations_config": {"enabled": True}}),
        TextContent(text="Who created each language mentioned in the documents?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.message.to_dict())

# Citations ride on the canonical TextContent: raw wire dicts in
# kwargs["citations"], typed (serialized) objects in kwargs["canonical_citations"].
for block in res.message.content:
    if block.kwargs.get("canonical_citations"):
        pprint({"citations": block.kwargs["canonical_citations"]})


{'attachments': [],
 'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EuMFCm0IDhgCKkCwGkxBLnBf2OaKV0g+hE2b1ytCHCd+nwp9LTyHgdWsZ1dbu912CL7IAGdE6/gbsfG/7NWnL+cqXdFGiuw+Sn3NMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nEgxZZ9pQ1rGbtbWJ0CoaDMv4kTdEWSiJjViMLSIwcRFDQ7hcsxJ10sVLdm/AQt6FuTJft6acydxtZrl1h1DszyslhNJdHCymJP2G+3uqKqMEbZW2h/9V9emmB9Yhl3DBTSeaI1qlhsaYie7Y7hra82We9K1gYTO4mQ5+2uAkJ+CLpqcXWzjHUhFgPbvXS81kIjAR2U/x2yTGq812Ch7tikZzaPTwnzO5EiIZUFPhkUQcsNX/b6CGcguhG1GApFXKW9DIgx1nJeCRwubYRw+SiXdrOuIhKhbTavMWH64sC2/gg2I9iavyii6qdZaRYt1Y8HUtTuJ+oyFjhnSKN+QQGkYWntMcWIOSDwh5qWPPnN1sDenJt/FhBf31pHqyGFTmD+9iO2t4ytf5cjNPtsI+PCc/+NLx9lI/VskuiMAlqPX3Xdr/wA0m5Pbt/+Ui7YF+Qugj2kHPSkllkESRnsPSEI9rf1Y9SkzK1lloCKNalX8ljqVYpPcjJZRZw87aNThi6Lbn3HT9W9kX2jTijCwkgP47dysh3E3zk7AzIsGTm3PxNOuKiGh3AatRlhfRi3skKiG/ILz0K8Gg3yf8ARZqLXQ/7jr2agB7sMN2N/wyFM2RB7N0YGNmkEHjt8UXJx/fARd7kWs3O8X2TYHjFL2jh8OD8vVK0hWhMI5DFIjjrijOma3Z/9rw3oxDunSEdrWFZD9kkf6Xxq0dzq3kWMrr